Concept:\
Contract Value:
- revenue from sales
- cost of purchase
- storage costs (monthly rent)
- Injection / withdrawal fees
- Transport costs

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

# ══════════════════════════════════════════════════════════════════
# PRICE ESTIMATOR 
# ══════════════════════════════════════════════════════════════════

def load_and_fit(path: str):
  """load csv and fit the fourier + trend model.
  Returns --> coeffs, base_date."""
  df = pd.read_csv(path, header=0, names=['Date', 'Price'])
  df['Date'] = pd.to_datetime(df['Date'], dayfirst=False)
  df = df.sort_values('Date').reset_index(drop=True)

  base_year = df['Date'].dt.year.iloc[0]
  base_month = df['Date'].dt.month.iloc[0]
  df['t'] = (df['Date'].dt.year - base_year) * 12 + \
            (df['Date'].dt.month - base_month)

  def build_features(t_vals):
    t = np.asarray(t_vals, dtype=float)
    cols = [np.ones_like(t), t,
            np.sin(2*np.pi*t/12), np.cos(2*np.pi*t/12),
            np.sin(4*np.pi*t/12), np.cos(4*np.pi*t/12)]
    return np.column_stack(cols)

  X = build_features(df['t'].values)
  coeffs = np.linalg.lstsq(X, df['Price'].values, rcond=None)[0]
  base_date = df['Date'].iloc[0].to_pydatetime()

  return coeffs, base_date, build_features

def get_price(date_input, coeffs, base_date, build_features) -> float:
    """Return model-estimated gas price ($/MMBtu) for any date."""
    dt       = pd.to_datetime(date_input)
    t        = (dt.year  - base_date.year)  * 12 + \
               (dt.month - base_date.month) + \
               (dt.day   - base_date.day)   / 30.44
    row      = build_features(np.array([t]))
    return float(row @ coeffs)

In [2]:
# ══════════════════════════════════════════════════════════════════
# CONTRACT PRICER
# ══════════════════════════════════════════════════════════════════

def price_contract(
    injection_dates:    list,
    withdrawal_dates:   list,
    injection_rate:     float,
    withdrawal_rate:    float,
    max_storage:        float,
    storage_cost_per_month: float,
    injection_cost_per_unit:   float = 0.0,
    withdrawal_cost_per_unit:  float = 0.0,
    transport_cost_per_trip:   float = 0.0,
    data_path:          str   = 'Nat_Gas__5___1_.csv',
    price_overrides:    dict  = None,
    verbose:            bool  = True,
) -> float:
    """
    Price a natural gas storage contract.

    PARAMETERS
    ----------
    injection_dates          : list of str/datetime — dates gas is purchased & injected
    withdrawal_dates         : list of str/datetime — dates gas is sold & withdrawn
    injection_rate           : MMBtu injected per day (caps volume purchased each date)
    withdrawal_rate          : MMBtu withdrawn per day (caps volume sold each date)
    max_storage              : maximum MMBtu that can be held in storage at any time
    storage_cost_per_month   : fixed $/month rent paid to the storage facility
    injection_cost_per_unit  : $/MMBtu fee charged on every MMBtu injected
    withdrawal_cost_per_unit : $/MMBtu fee charged on every MMBtu withdrawn
    transport_cost_per_trip  : flat $ fee charged per injection OR withdrawal trip
    data_path                : path to the natural gas price CSV
    price_overrides          : optional dict {date_str: price} to hard-set prices
                               (useful for testing or when desk has live quotes)
    verbose                  : if True, print a full cash-flow breakdown

    RETURNS
    -------
    float : net contract value in dollars (positive = profitable to the buyer)

    LOGIC
    -----
    1. Convert all dates; sort injections and withdrawals chronologically.
    2. For each injection date:
         - volume = min(injection_rate, remaining_storage_capacity)
         - cash outflow = volume × buy_price + injection_fee + transport_fee
         - update stored volume
    3. For each withdrawal date:
         - volume = min(withdrawal_rate, current_stored_volume)
         - cash inflow  = volume × sell_price − withdrawal_fee − transport_fee
         - update stored volume
    4. Storage cost = monthly_rent × number_of_months_in_storage
         (counted as whole months between first injection and last withdrawal)
    5. Contract value = total inflows − total outflows − total storage cost
    """

    if price_overrides is None:
        price_overrides = {}

    # ── Load price model ─────────────────────────────────────────
    coeffs, base_date, build_features = load_and_fit(data_path)

    def lookup_price(d):
        key = pd.to_datetime(d).strftime('%Y-%m-%d')
        if key in price_overrides:
            return price_overrides[key]
        return get_price(d, coeffs, base_date, build_features)

    # ── Parse and sort dates ──────────────────────────────────────
    inj_dates  = sorted(pd.to_datetime(injection_dates))
    with_dates = sorted(pd.to_datetime(withdrawal_dates))

    all_dates  = sorted(inj_dates + with_dates)
    start_date = all_dates[0]
    end_date   = all_dates[-1]

    # ── Simulate cash flows ───────────────────────────────────────
    stored_volume   = 0.0
    total_purchase  = 0.0
    total_revenue   = 0.0
    total_inj_fee   = 0.0
    total_with_fee  = 0.0
    total_transport = 0.0
    ledger          = []

    for d in inj_dates:
        available_space = max_storage - stored_volume
        volume          = min(injection_rate, available_space)

        if volume <= 0:
            ledger.append({
                'Date': d.strftime('%Y-%m-%d'),
                'Action': 'INJECT (SKIPPED — storage full)',
                'Volume (MMBtu)': 0,
                'Price ($/MMBtu)': '-',
                'Cash Flow ($)': 0,
            })
            continue

        price    = lookup_price(d)
        purchase = volume * price
        inj_fee  = volume * injection_cost_per_unit
        transp   = transport_cost_per_trip
        net_flow = -(purchase + inj_fee + transp)   # outflow → negative

        stored_volume  += volume
        total_purchase += purchase
        total_inj_fee  += inj_fee
        total_transport+= transp

        ledger.append({
            'Date': d.strftime('%Y-%m-%d'),
            'Action': 'INJECT',
            'Volume (MMBtu)': round(volume, 2),
            'Price ($/MMBtu)': round(price, 4),
            'Cash Flow ($)': round(net_flow, 2),
        })

    for d in with_dates:
        volume = min(withdrawal_rate, stored_volume)

        if volume <= 0:
            ledger.append({
                'Date': d.strftime('%Y-%m-%d'),
                'Action': 'WITHDRAW (SKIPPED — storage empty)',
                'Volume (MMBtu)': 0,
                'Price ($/MMBtu)': '-',
                'Cash Flow ($)': 0,
            })
            continue

        price     = lookup_price(d)
        revenue   = volume * price
        with_fee  = volume * withdrawal_cost_per_unit
        transp    = transport_cost_per_trip
        net_flow  = revenue - with_fee - transp     # inflow → positive

        stored_volume   -= volume
        total_revenue   += revenue
        total_with_fee  += with_fee
        total_transport += transp

        ledger.append({
            'Date': d.strftime('%Y-%m-%d'),
            'Action': 'WITHDRAW',
            'Volume (MMBtu)': round(volume, 2),
            'Price ($/MMBtu)': round(price, 4),
            'Cash Flow ($)': round(net_flow, 2),
        })

    # ── Storage cost: whole months from first to last event ───────
    months_stored  = (end_date.year  - start_date.year) * 12 + \
                     (end_date.month - start_date.month)
    months_stored  = max(months_stored, 1)   # at least 1 month
    total_storage  = storage_cost_per_month * months_stored

    # ── Final valuation ───────────────────────────────────────────
    contract_value = (total_revenue
                      - total_purchase
                      - total_inj_fee
                      - total_with_fee
                      - total_transport
                      - total_storage)

    # ── Verbose output ────────────────────────────────────────────
    if verbose:
        print("=" * 62)
        print("  NATURAL GAS STORAGE CONTRACT — VALUATION REPORT")
        print("=" * 62)

        print(f"\n  {'Date':<14} {'Action':<34} {'Vol':>10} {'Price':>10} {'Flow ($)':>12}")
        print(f"  {'-'*13} {'-'*33} {'-'*10} {'-'*10} {'-'*12}")
        for row in sorted(ledger, key=lambda x: x['Date']):
            price_str = f"{row['Price ($/MMBtu)']:.4f}" if isinstance(row['Price ($/MMBtu)'], float) else '-'
            print(f"  {row['Date']:<14} {row['Action']:<34} "
                  f"{row['Volume (MMBtu)']:>10.2f} "
                  f"{price_str:>10} "
                  f"{row['Cash Flow ($)']:>12,.2f}")

        print(f"\n  {'─'*60}")
        print(f"  {'Total Revenue (sales):':<38} ${total_revenue:>14,.2f}")
        print(f"  {'Total Purchase Cost:':<38} ${-total_purchase:>14,.2f}")
        print(f"  {'Injection Fees:':<38} ${-total_inj_fee:>14,.2f}")
        print(f"  {'Withdrawal Fees:':<38} ${-total_with_fee:>14,.2f}")
        print(f"  {'Transport Costs:':<38} ${-total_transport:>14,.2f}")
        print(f"  {'Storage Cost ({} months × ${:,.0f}/mo):'.format(months_stored, storage_cost_per_month):<38} ${-total_storage:>14,.2f}")
        print(f"  {'─'*60}")
        print(f"  {'CONTRACT VALUE:':<38} ${contract_value:>14,.2f}")
        print(f"  {'═'*60}")
        verdict = "PROFITABLE ✓" if contract_value > 0 else "LOSS-MAKING ✗" if contract_value < 0 else "BREAK-EVEN"
        print(f"  Verdict: {verdict}")
        print()

    return round(contract_value, 2)

In [3]:
# ══════════════════════════════════════════════════════════════════
# TEST CASES
# ══════════════════════════════════════════════════════════════════

if __name__ == '__main__':

    DATA = '../data/Nat_Gas (5) (1).csv'

    # ──────────────────────────────────────────────────────────────
    # TEST 1: Simple single buy/sell
    # Buy in summer (low price), sell in winter (high price)
    # Expected: profitable — classic seasonal arb
    # ──────────────────────────────────────────────────────────────
    print("\n" + "█"*62)
    print("  TEST 1 — Simple Summer Buy / Winter Sell")
    print("█"*62)
    v1 = price_contract(
        injection_dates          = ['2024-06-30'],
        withdrawal_dates         = ['2024-12-31'],
        injection_rate           = 1_000_000,
        withdrawal_rate          = 1_000_000,
        max_storage              = 1_000_000,
        storage_cost_per_month   = 100_000,
        injection_cost_per_unit  = 0.0,
        withdrawal_cost_per_unit = 0.0,
        transport_cost_per_trip  = 0.0,
        data_path                = DATA,
    )

    # ──────────────────────────────────────────────────────────────
    # TEST 2: With all fees included
    # Same trade but now with injection/withdrawal + transport fees
    # Expected: lower value than Test 1
    # ──────────────────────────────────────────────────────────────
    print("\n" + "█"*62)
    print("  TEST 2 — Same Trade + Full Fees")
    print("█"*62)
    v2 = price_contract(
        injection_dates          = ['2024-06-30'],
        withdrawal_dates         = ['2024-12-31'],
        injection_rate           = 1_000_000,
        withdrawal_rate          = 1_000_000,
        max_storage              = 1_000_000,
        storage_cost_per_month   = 100_000,
        injection_cost_per_unit  = 0.01,     # $10K per 1M MMBtu
        withdrawal_cost_per_unit = 0.01,
        transport_cost_per_trip  = 50_000,   # $50K per trip
        data_path                = DATA,
    )

    # ──────────────────────────────────────────────────────────────
    # TEST 3: Multiple injection & withdrawal dates
    # Buy across 3 summer months, sell across 3 winter months
    # ──────────────────────────────────────────────────────────────
    print("\n" + "█"*62)
    print("  TEST 3 — Multiple Injections & Withdrawals")
    print("█"*62)
    v3 = price_contract(
        injection_dates          = ['2024-05-31', '2024-06-30', '2024-07-31'],
        withdrawal_dates         = ['2024-11-30', '2024-12-31', '2025-01-31'],
        injection_rate           = 400_000,      # 400K MMBtu per injection date
        withdrawal_rate          = 400_000,
        max_storage              = 1_000_000,    # can hold up to 1M
        storage_cost_per_month   = 100_000,
        injection_cost_per_unit  = 0.005,
        withdrawal_cost_per_unit = 0.005,
        transport_cost_per_trip  = 25_000,
        data_path                = DATA,
    )

    # ──────────────────────────────────────────────────────────────
    # TEST 4: Storage cap binding
    # Try to inject more than max_storage — model should cap volumes
    # ──────────────────────────────────────────────────────────────
    print("\n" + "█"*62)
    print("  TEST 4 — Storage Cap Binding (injection_rate > max_storage)")
    print("█"*62)
    v4 = price_contract(
        injection_dates          = ['2024-06-30', '2024-07-31'],
        withdrawal_dates         = ['2024-12-31'],
        injection_rate           = 700_000,       # would exceed 1M cap on 2nd injection
        withdrawal_rate          = 1_500_000,
        max_storage              = 1_000_000,
        storage_cost_per_month   = 80_000,
        injection_cost_per_unit  = 0.0,
        withdrawal_cost_per_unit = 0.0,
        transport_cost_per_trip  = 0.0,
        data_path                = DATA,
    )

    # ──────────────────────────────────────────────────────────────
    # TEST 5: Using price_overrides (desk has live quotes)
    # ──────────────────────────────────────────────────────────────
    print("\n" + "█"*62)
    print("  TEST 5 — Manual Price Overrides (desk live quotes)")
    print("█"*62)
    v5 = price_contract(
        injection_dates          = ['2024-06-30'],
        withdrawal_dates         = ['2025-01-31'],
        injection_rate           = 1_000_000,
        withdrawal_rate          = 1_000_000,
        max_storage              = 1_000_000,
        storage_cost_per_month   = 100_000,
        injection_cost_per_unit  = 0.01,
        withdrawal_cost_per_unit = 0.01,
        transport_cost_per_trip  = 50_000,
        price_overrides          = {
            '2024-06-30': 10.50,    # desk's live buy quote
            '2025-01-31': 13.50,    # desk's live sell quote
        },
        data_path                = DATA,
    )

    # ── Summary ───────────────────────────────────────────────────
    print("\n" + "═"*62)
    print("  SUMMARY OF ALL TEST CASES")
    print("═"*62)
    print(f"  Test 1 (Simple buy/sell, no fees):      ${v1:>12,.2f}")
    print(f"  Test 2 (Same + all fees):               ${v2:>12,.2f}")
    print(f"  Test 3 (Multi-date injections/sales):   ${v3:>12,.2f}")
    print(f"  Test 4 (Storage cap binding):           ${v4:>12,.2f}")
    print(f"  Test 5 (Manual price overrides):        ${v5:>12,.2f}")
    print("═"*62)


██████████████████████████████████████████████████████████████
  TEST 1 — Simple Summer Buy / Winter Sell
██████████████████████████████████████████████████████████████


C:\Users\vinay\AppData\Local\Temp\ipykernel_62940\835299977.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Date'], dayfirst=False)


  NATURAL GAS STORAGE CONTRACT — VALUATION REPORT

  Date           Action                                    Vol      Price     Flow ($)
  ------------- --------------------------------- ---------- ---------- ------------
  2024-06-30     INJECT                             1000000.00    11.5567 -11,556,657.94
  2024-12-31     WITHDRAW                           1000000.00    12.9800 12,980,002.81

  ────────────────────────────────────────────────────────────
  Total Revenue (sales):                 $ 12,980,002.81
  Total Purchase Cost:                   $-11,556,657.94
  Injection Fees:                        $         -0.00
  Withdrawal Fees:                       $         -0.00
  Transport Costs:                       $         -0.00
  Storage Cost (6 months × $100,000/mo): $   -600,000.00
  ────────────────────────────────────────────────────────────
  CONTRACT VALUE:                        $    823,344.87
  ════════════════════════════════════════════════════════════
  Verdict: 

C:\Users\vinay\AppData\Local\Temp\ipykernel_62940\835299977.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return float(row @ coeffs)
C:\Users\vinay\AppData\Local\Temp\ipykernel_62940\835299977.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Date'], dayfirst=False)
C:\Users\vinay\AppData\Local\Temp\ipykernel_62940\835299977.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return float(row @ coeffs)
C:\Users\vinay\AppData\Local\Temp\ipykernel_62940\8352999